# FPL Points Prediction - Data Exploration

This notebook explores the dataset we will use to predict Fantasy Premier League (FPL) player points.

**Goal:** Understand the structure of the data, the distribution of our target variable, and which features
might be useful for prediction.

**Dataset:** `data/processed/features.csv` - one row per player per gameweek, with rolling statistics
and a `target_points` column we want to predict.

In [ ]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Use a clean, readable style for all plots
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

---
## 1. Overview

Let's load the data and take a first look at what we're working with.

In [ ]:
# Load the dataset
df = pd.read_csv('../data/processed/features.csv')

print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Gameweek range: GW{df['gameweek'].min()} to GW{df['gameweek'].max()}")
print(f"Unique players: {df['player_id'].nunique()}")
print()

In [ ]:
# Column types - notice the mix of categorical and numeric features
df.dtypes

In [ ]:
# A few sample rows to get a feel for the data
df.head(10)

In [ ]:
# Summary statistics for all numeric columns
df.describe().round(2)

In [ ]:
# Check for missing values
missing = df.isnull().sum()
if missing.sum() == 0:
    print("No missing values in the dataset.")
else:
    print("Missing values:")
    print(missing[missing > 0])

---
## 2. Target Distribution

The column we want to predict is `target_points` - the actual FPL points a player scored in a given gameweek.

In FPL, most players score very few points in any given week. A typical return is 1-2 points
(just for playing). Big scores (10+) are rare events. This makes prediction inherently difficult
because the target distribution is **heavily right-skewed**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of target points
axes[0].hist(df['target_points'], bins=range(-5, 25), edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('Points Scored')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Target Points')
axes[0].axvline(df['target_points'].mean(), color='red', linestyle='--', label=f"Mean: {df['target_points'].mean():.1f}")
axes[0].axvline(df['target_points'].median(), color='orange', linestyle='--', label=f"Median: {df['target_points'].median():.1f}")
axes[0].legend()

# Show the percentage breakdown for common point values
pct = df['target_points'].value_counts(normalize=True).sort_index() * 100
low_pts = pct.loc[pct.index.isin([1, 2])].sum()
axes[1].bar(pct.index, pct.values, color='steelblue', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Points Scored')
axes[1].set_ylabel('Percentage of All Rows (%)')
axes[1].set_title('Percentage Breakdown by Points Value')
axes[1].set_xlim(-5, 20)

plt.tight_layout()
plt.show()

print(f"Rows with 1-2 points: {low_pts:.1f}% of the dataset")
print(f"Rows with 0 or fewer points: {pct.loc[pct.index <= 0].sum():.1f}%")
print(f"Rows with 10+ points: {pct.loc[pct.index >= 10].sum():.1f}%")

**Key insight:** Around 58% of all observations score just 1-2 points. The distribution has a long right
tail - big hauls of 10+ points are rare but are exactly what FPL managers want to predict.

This heavy skew means:
- A model that always predicts ~2 points would be "right" most of the time but completely useless.
- We need the model to capture the *differences* between players, even though those differences are small.
- Standard regression metrics like MAE will be dominated by the majority of low-scoring rows.

---
## 3. Points by Position

FPL has four positions, each with different scoring rules:
- **GK** (Goalkeepers): Points for saves and clean sheets
- **DEF** (Defenders): Bonus for clean sheets, goals worth more
- **MID** (Midfielders): Balanced scoring, goals and assists
- **FWD** (Forwards): Mainly goals and assists

In [ ]:
# Order positions in the standard FPL order
position_order = ['GK', 'DEF', 'MID', 'FWD']
position_colors = {'GK': '#2ecc71', 'DEF': '#3498db', 'MID': '#e74c3c', 'FWD': '#f39c12'}

fig, ax = plt.subplots(figsize=(10, 6))

# Create box plot
data_by_pos = [df[df['position'] == pos]['target_points'] for pos in position_order]
bp = ax.boxplot(data_by_pos, labels=position_order, patch_artist=True, widths=0.6)

# Color each box
for patch, pos in zip(bp['boxes'], position_order):
    patch.set_facecolor(position_colors[pos])
    patch.set_alpha(0.7)

ax.set_xlabel('Position')
ax.set_ylabel('Points Scored')
ax.set_title('Points Distribution by Position')

plt.tight_layout()
plt.show()

# Summary stats per position
print("Points summary by position:")
print(df.groupby('position')['target_points'].agg(['mean', 'median', 'std', 'count']).round(2)
      .reindex(position_order))

**Observations:**
- **Midfielders** tend to have the highest variance in points. They can score through goals, assists,
  clean sheets (if they play deep), and bonus points - giving them more routes to big hauls.
- **Forwards** are volatile - they either score (and get big points) or return very little.
  Their upside is high but so is the downside.
- **Goalkeepers** are the most consistent (low variance). Their floor is higher (save points)
  but their ceiling is lower.
- **Defenders** sit in between - clean sheets provide a nice bonus, but it's binary.

This tells us that **position is an important feature** and our model should probably
treat positions differently.

---
## 4. Feature Correlation

Let's look at how each numeric feature correlates with our target. High correlation does not mean
causation, but it gives us a starting point for understanding which features carry signal.

In [ ]:
# Select numeric features (exclude IDs and categorical columns)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
exclude_cols = ['player_id']  # IDs are not features
feature_cols = [c for c in numeric_cols if c not in exclude_cols]

# Compute correlation matrix
corr = df[feature_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')

# Add labels
ax.set_xticks(range(len(feature_cols)))
ax.set_yticks(range(len(feature_cols)))
ax.set_xticklabels(feature_cols, rotation=45, ha='right', fontsize=10)
ax.set_yticklabels(feature_cols, fontsize=10)

# Add correlation values in cells
for i in range(len(feature_cols)):
    for j in range(len(feature_cols)):
        val = corr.iloc[i, j]
        color = 'white' if abs(val) > 0.5 else 'black'
        ax.text(j, i, f"{val:.2f}", ha='center', va='center', fontsize=7, color=color)

plt.colorbar(im, ax=ax, shrink=0.8, label='Pearson Correlation')
ax.set_title('Feature Correlation Heatmap', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# Which features correlate most with target_points?
target_corr = corr['target_points'].drop('target_points').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['steelblue' if v > 0 else 'salmon' for v in target_corr.values]
ax.barh(target_corr.index, target_corr.values, color=colors, edgecolor='black', alpha=0.7)
ax.set_xlabel('Correlation with target_points')
ax.set_title('Feature Correlation with Target Points')
ax.axvline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

print("Top correlations with target_points:")
print(target_corr.round(3).to_string())

**Key findings:**
- The strongest predictors are typically **form-based features** (form_3gw, form_5gw) and
  **ICT index** - these capture recent performance momentum.
- **Minutes played** (avg_minutes_3gw) is important - players who don't play get 0 points.
- Season-level stats (goals_per_90, assists_per_90) provide a baseline of player quality.
- No single feature has a very high correlation, which tells us prediction will require
  combining multiple signals.

---
## 5. Rolling Form Analysis

The `form_3gw` feature is the player's average points over the last 3 gameweeks. In theory,
recent form should predict future performance. Let's see how strong this signal actually is.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter plot: form_3gw vs target_points
axes[0].scatter(df['form_3gw'], df['target_points'], alpha=0.05, s=10, color='steelblue')
axes[0].set_xlabel('3-GW Rolling Form (avg points)')
axes[0].set_ylabel('Actual Points (target)')
axes[0].set_title('3-GW Form vs Actual Points')

# Add a trend line
z = np.polyfit(df['form_3gw'], df['target_points'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['form_3gw'].min(), df['form_3gw'].max(), 100)
axes[0].plot(x_line, p(x_line), 'r--', linewidth=2, label=f'Trend (slope={z[0]:.2f})')
axes[0].legend()

# Binned version to see the trend more clearly
df['form_bin'] = pd.cut(df['form_3gw'], bins=10)
binned = df.groupby('form_bin', observed=True)['target_points'].agg(['mean', 'std', 'count'])
binned = binned[binned['count'] >= 10]  # Only show bins with enough data

bin_labels = [f"{interval.mid:.1f}" for interval in binned.index]
axes[1].bar(range(len(binned)), binned['mean'], yerr=binned['std'], 
            color='steelblue', edgecolor='black', alpha=0.7, capsize=3)
axes[1].set_xticks(range(len(binned)))
axes[1].set_xticklabels(bin_labels, rotation=45)
axes[1].set_xlabel('Form Bin (midpoint)')
axes[1].set_ylabel('Mean Actual Points')
axes[1].set_title('Mean Points by Form Bin (with Std Dev)')

plt.tight_layout()
plt.show()

# Clean up temp column
df.drop('form_bin', axis=1, inplace=True)

print(f"Correlation between form_3gw and target_points: {df['form_3gw'].corr(df['target_points']):.3f}")

**Observations:**
- The signal is **noisy but directional** - players in better form do tend to score more points,
  but the scatter is wide.
- The binned view shows the trend more clearly: mean points increases with form, but the standard
  deviation is large in every bin.
- This is typical of sports prediction: there is real signal in form, but individual gameweek
  outcomes are highly stochastic (a player can hit the post 3 times and get 2 points).

---
## 6. Home vs Away

Home advantage is a well-known effect in football. Let's see if it shows up in FPL points too.

In [ ]:
home_away = df.groupby('was_home')['target_points'].agg(['mean', 'median', 'std', 'count'])
home_away.index = ['Away', 'Home']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(['Away', 'Home'], home_away['mean'], 
              color=['#e74c3c', '#2ecc71'], edgecolor='black', alpha=0.7, width=0.5)

# Add value labels on bars
for bar, val in zip(bars, home_away['mean']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, 
            f"{val:.2f}", ha='center', va='bottom', fontsize=13, fontweight='bold')

ax.set_ylabel('Mean Points')
ax.set_title('Mean Points: Home vs Away')
ax.set_ylim(0, home_away['mean'].max() * 1.2)

plt.tight_layout()
plt.show()

print("Home vs Away summary:")
print(home_away.round(2))
diff = home_away.loc['Home', 'mean'] - home_away.loc['Away', 'mean']
print(f"\nHome advantage: +{diff:.2f} points on average")

Playing at home does give a measurable boost in FPL points. This makes sense - home teams
tend to score more goals and concede fewer, which translates directly into FPL points via
goals, assists, and clean sheets. The `was_home` feature should be useful for our model.

---
## 7. Fixture Difficulty

FPL assigns each fixture a difficulty rating (typically 1-5, where 5 is the hardest).
We'd expect players to score fewer points in harder fixtures.

In [ ]:
fdr = df.groupby('fixture_difficulty')['target_points'].agg(['mean', 'median', 'count'])

fig, ax = plt.subplots(figsize=(9, 5))

# Color gradient from green (easy) to red (hard)
n_bars = len(fdr)
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, n_bars))

bars = ax.bar(fdr.index.astype(str), fdr['mean'], color=colors, edgecolor='black', alpha=0.8)

# Add value labels
for bar, val in zip(bars, fdr['mean']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f"{val:.2f}", ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xlabel('Fixture Difficulty Rating')
ax.set_ylabel('Mean Points')
ax.set_title('Mean Points by Fixture Difficulty')
ax.set_ylim(0, fdr['mean'].max() * 1.2)

plt.tight_layout()
plt.show()

print("Points by fixture difficulty:")
print(fdr.round(2))

As expected, players tend to score fewer points in harder fixtures. The effect is modest but
consistent - another useful signal for our model. FPL managers intuitively know this: you
want your players facing weaker opposition.

---
## 8. Train/Test Split

For time-series problems like this, we **must** split temporally - training on past gameweeks
and testing on future ones. Using a random split would leak future information and give
unrealistically good results.

Our split:
- **Training set:** Gameweeks 4-30
- **Test set:** Gameweeks 31+

In [ ]:
train = df[df['gameweek'] <= 30]
test = df[df['gameweek'] > 30]

print(f"Training set: GW{train['gameweek'].min()}-{train['gameweek'].max()}")
print(f"  Rows: {len(train):,}")
print(f"  Unique players: {train['player_id'].nunique()}")
print(f"  Mean target: {train['target_points'].mean():.2f}")
print()
print(f"Test set: GW{test['gameweek'].min()}-{test['gameweek'].max()}")
print(f"  Rows: {len(test):,}")
print(f"  Unique players: {test['player_id'].nunique()}")
print(f"  Mean target: {test['target_points'].mean():.2f}")

In [ ]:
# Visualize the split
gw_counts = df.groupby('gameweek').size()

fig, ax = plt.subplots(figsize=(12, 5))

train_gw = gw_counts[gw_counts.index <= 30]
test_gw = gw_counts[gw_counts.index > 30]

ax.bar(train_gw.index, train_gw.values, color='steelblue', edgecolor='black', alpha=0.7, label='Train')
ax.bar(test_gw.index, test_gw.values, color='#e74c3c', edgecolor='black', alpha=0.7, label='Test')
ax.axvline(30.5, color='black', linestyle='--', linewidth=2, label='Split point (GW 30)')

ax.set_xlabel('Gameweek')
ax.set_ylabel('Number of Player-Rows')
ax.set_title('Temporal Train/Test Split')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Check that train and test target distributions are similar
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train['target_points'], bins=range(-5, 25), edgecolor='black', alpha=0.7, 
             color='steelblue', density=True)
axes[0].set_title(f'Train Target Distribution (n={len(train):,})')
axes[0].set_xlabel('Points')
axes[0].set_ylabel('Density')

axes[1].hist(test['target_points'], bins=range(-5, 25), edgecolor='black', alpha=0.7, 
             color='#e74c3c', density=True)
axes[1].set_title(f'Test Target Distribution (n={len(test):,})')
axes[1].set_xlabel('Points')
axes[1].set_ylabel('Density')

plt.tight_layout()
plt.show()

print(f"Train mean: {train['target_points'].mean():.2f}, std: {train['target_points'].std():.2f}")
print(f"Test mean:  {test['target_points'].mean():.2f}, std: {test['target_points'].std():.2f}")

The test set is much smaller than the training set (243 vs ~8,225 rows). This is a realistic
scenario - in practice, we'd be predicting one gameweek at a time. The target distributions
look broadly similar, which is a good sign that the split is reasonable.

---
## 9. Summary - Key Takeaways

Here's what we've learned about this prediction problem:

### Why this problem is hard

1. **Heavily skewed target:** ~58% of rows are 1-2 points. The "interesting" outcomes (big hauls)
   are rare events, making them hard to predict.

2. **Noisy signal:** Even the best individual features (form, ICT index) have modest correlation
   with the target. Football has a huge random component.

3. **Small test set:** With only ~243 test rows, our evaluation will have high variance.
   A model might look good or bad partly due to luck.

4. **Non-stationarity:** Player form, injuries, and team dynamics change over the season.
   Patterns from early gameweeks may not hold later.

### What gives us hope

1. **Form features carry signal:** Rolling averages (form_3gw, form_5gw) do predict future
   performance, even if noisily.

2. **Contextual features help:** Home/away, fixture difficulty, and team-level stats all show
   meaningful relationships with points.

3. **Position matters:** Different positions have different scoring profiles, which the model
   can learn.

4. **Multiple weak signals:** While no single feature is highly predictive, combining many
   weak signals is exactly what ML is good at.

### Next steps

- Train baseline models (linear regression, random forest) and compare to a "predict the mean" baseline
- Experiment with feature engineering (interaction terms, more rolling windows)
- Try gradient boosting (XGBoost/LightGBM) which tends to work well with tabular data like this
- Consider position-specific models or adding position as a feature